 ### Adaptive RAG
 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [3]:
##Build Index

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embd = HuggingFaceEmbeddings()

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

## Add to vectorstore
vectorstore = FAISS.from_documents(
    documents = doc_splits,
    embedding = HuggingFaceEmbeddings()
)

retriever = vectorstore.as_retriever()

/Users/shivanshmishra2701gmail.com/Desktop/AgenticWorkspace/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2b/7vsbn3nj3lzcywpjqw6q6rz00000gn/T/ipykernel_1336/3964389321.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10719.13it/s]


In [4]:
## Router 
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field


##DataModel
class RouteQuery(BaseModel):
    datasource: Literal["vectorstore","websearch"] = Field(
        ...,
        description = "Given a user question choose to route it to websearch or a vectorstore"
    )


llm = ChatGroq(model = "llama-3.3-70b-versatile",temperature=0)
structured_llm_router = llm.with_structured_output(RouteQuery)

## Prompt
system = """
you are an expert at routing a user question to a vectorstore or websearch.
The vectorstore contains documents related to agents, prompt engineeingm and adversial attacks.
use the vectorstore for questions on these topics.Otherwise, use web-search."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system),
        ("human","{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

print(
    question_router.invoke(
        {"question":"Who won the cricket world cup 2023"}
    )
)

datasource='websearch'


In [5]:
print(question_router.invoke({"question":"What are the types of agent memory"}))

datasource='vectorstore'


In [ ]:
## Retrieval Grader
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

#Data Model
class GradeDocuments(BaseModel):
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no' "
    )

#LLM with function call
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

#Prompt
system = """You are grader assesing relvance of a retrieved document to a user queston. \n
If the document contains keyword(s) or semantic meaning related to the question, grade it as relevant.\n
Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question. """
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system),
        ("human","Retrieved document:\n\n {document} \n\n User Question: {question}"),
    ]
)
## Chain the prompt with the LLM

retrieval_grader = grade_prompt | structured_llm_grader
question = "agent memory"
docs = retriever.invoke(question)
doc_txt = docs[1].page_content
print(retrieval_grader.invoke({"question":question,"document":doc_txt}))


binary_score='yes'
